# Upload a Laminar Flame Speed (LFS) experiment via the SciExpeM API

This example uploads a **laminar burning velocity** measurement (equivalence-ratio sweep)
using three input files taken from the paper:

- `metadataPaper.txt`  &rarr; bibliographic reference (`FilePaper`)
- `expdata_OME2_1.txt`  &rarr; the experimental data (`Eq.R` vs `LFS`)
- `input_OME2_1.dic`    &rarr; the OpenSMOKE `FlameSpeed` input file

Fuel: **OME2** (C4H10O3) in air, inlet T = 473 K, P = 1.03 bar.

The experiment is assembled from client-side model objects and sent with
`insertElement`, exactly like the IDT example. The only LFS-specific parts are:
- `reactor='flame'`, `experiment_type='laminar burning velocity measurement'`
- data columns named **`equivalence ratio`**, **`laminar burning velocity`**, and
  one **`composition`** column per species (mole fraction at each phi point).
  This name set is what lets SciExpeM auto-match the LFS interpreter (EXP24,
  solver `PremixedLaminarFlame1D`).
- for LFS set `t_inf = t_sup =` the inlet temperature.

### Connect

In [ ]:
from SciExpeM_API.SciExpeM import SciExpeM
from SciExpeM_API.Models import *

# Fill in your credentials (or use token=...)
my_sciexpem = SciExpeM(username=USERNAME, password=PASSWORD)

### Point to the three input files

In [ ]:
# Edit these paths to where you saved the files
metadata_path = 'metadataPaper.txt'
expdata_path  = 'expdata_OME2_1.txt'
input_path    = 'input_OME2_1.dic'

### Read the experimental data (Eq.R vs LFS)

The data file has two whitespace-separated columns with a header line
`Eq.R  LFS`. LFS here is in **cm/s**.

In [ ]:
phi = []
lfs = []
with open(expdata_path) as f:
    header = f.readline()            # skip 'Eq.R  LFS'
    for line in f:
        line = line.strip()
        if not line:
            continue
        x, y = line.split()
        phi.append(float(x))
        lfs.append(float(y))

print('phi:', phi)
print('LFS [cm/s]:', lfs)

### Compute the per-point composition from phi + stoichiometry

EXP24 (the LFS interpreter) expects the data-column name set
`{composition, equivalence ratio, laminar burning velocity}`. We therefore add
one `composition` column (mole fraction) per species: **OME2, O2, N2**.

From `input_OME2_1.dic`: fuel = `OME2 1`, oxidizer = air `O2 0.21 / N2 0.79`.

**OME2 = C4H10O3**, so the stoichiometric O2 per mol fuel is
`C + H/4 - O/2 = 4 + 10/4 - 3/2 = 5`.

At equivalence ratio phi (air, O2:N2 = 0.21:0.79):
`n_fuel = phi`, `n_O2 = 5`, `n_N2 = 5 * (0.79/0.21)`, then normalise to mole fractions.
Edit `a_stoich` / oxidizer split if your fuel or oxidizer differs.

In [ ]:
a_stoich = 5.0          # mol O2 per mol OME2 (C4H10O3)
o2_frac, n2_frac = 0.21, 0.79

X_OME2, X_O2, X_N2 = [], [], []
for p in phi:
    n_fuel = p
    n_O2   = a_stoich
    n_N2   = a_stoich * (n2_frac / o2_frac)
    tot = n_fuel + n_O2 + n_N2
    X_OME2.append(n_fuel / tot)
    X_O2.append(n_O2 / tot)
    X_N2.append(n_N2 / tot)

print('X_OME2:', [round(x, 6) for x in X_OME2])
print('X_O2  :', [round(x, 6) for x in X_O2])
print('X_N2  :', [round(x, 6) for x in X_N2])

### Data columns

All columns live in the same datagroup (`dg1`). The three column *names*
`equivalence ratio`, `laminar burning velocity`, `composition` are what the
interpreter matches on (do not rename them). The `composition` columns carry
the species via `species=` so SciExpeM knows which mole fraction is which.

In [ ]:
# species objects (needed for the composition columns and initial species)
sp_OME2 = my_sciexpem.filterDatabase(model_name='Species', preferredKey='OME2')[0]
sp_O2   = my_sciexpem.filterDatabase(model_name='Species', preferredKey='O2')[0]
sp_N2   = my_sciexpem.filterDatabase(model_name='Species', preferredKey='N2')[0]

d_phi = DataColumn(name='equivalence ratio', units='unitless',
                   dg_id='dg1', dg_label='experimental_data',
                   data=phi, source_type='reported')

d_lfs = DataColumn(name='laminar burning velocity', units='cm/s',
                   dg_id='dg1', dg_label='experimental_data',
                   data=lfs, source_type='reported')

# composition columns: one per species. NB the client kwarg is `species_object`
# and it takes a LIST of Species objects.
d_c_fuel = DataColumn(name='composition', units='mole fraction',
                      dg_id='dg1', dg_label='experimental_data',
                      data=X_OME2, source_type='calculated', species_object=[sp_OME2])

d_c_o2 = DataColumn(name='composition', units='mole fraction',
                    dg_id='dg1', dg_label='experimental_data',
                    data=X_O2, source_type='calculated', species_object=[sp_O2])

d_c_n2 = DataColumn(name='composition', units='mole fraction',
                    dg_id='dg1', dg_label='experimental_data',
                    data=X_N2, source_type='calculated', species_object=[sp_N2])

datacols = [d_phi, d_lfs, d_c_fuel, d_c_o2, d_c_n2]

### Common properties & initial species

From the `.dic`: inlet **T = 473 K**, **P = 1.03 bar**. Composition varies point
by point, so it is carried in the `composition` data columns above rather than as
a single initial composition; we still declare the initial species for the fuel/
oxidizer with `configuration='premixed'`.

In [ ]:
T_in = 473.0   # K  (from @Temperature of inlet-stream)
P    = 1.03    # bar (from @Pressure)

cp_T = CommonProperty(name='temperature', units='K', value=str(T_in), source_type='reported')
cp_P = CommonProperty(name='pressure',    units='bar', value=str(P),  source_type='reported')
commonprop = [cp_T, cp_P]

# For a phi-sweep LFS experiment the composition is per-point (carried in the
# 'composition' data columns), so there is NO single initial composition:
# leave initial_species empty. (Matches existing phi-based LFS experiments.)
inspecies = []

### FilePaper (from metadataPaper.txt)

`metadataPaper.txt` is a free-text reference. Here we fill the fields from that
paper. Easiest is to create/copy the reference from the SciExpeM web UI, but the
values below come straight from the file.

In [ ]:
file_paper = FilePaper(
    reference_doi='10.1016/j.proci.2022.09.070',
    author='Wei Li, Qilong Fang, Jianguo Zhang, Yanpak Chow, Lili Ye, Yuyang Li',
    title='Role of CH2O moiety on laminar burning velocities of oxymethylene ethers (OMEn): A case study of dimethyl ether, OME1 and OME2',
    year='2023',
    journal='Proceedings of the Combustion Institute',
    volume='39',
    page='795-804',
    description='Wei Li et al., Proceedings of the Combustion Institute, Volume 39, Issue 1, 2023, Pages 795-804, https://doi.org/10.1016/j.proci.2022.09.070',
)

### OpenSMOKE input file

In [ ]:
with open(input_path) as f:
    inputstr = f.read()

### Assemble the experiment

For LFS the characteristics are: `phi_inf/phi_sup` = phi range, `p_inf/p_sup` = P,
and **`t_inf = t_sup =` the inlet temperature** (the flame is characterised by its
fresh-gas inlet T, not a range).

In [ ]:
e = Experiment(
    reactor='flame',
    experiment_type='laminar burning velocity measurement',
    reactor_modes=['premixed'],
    file_paper=file_paper,
    data_columns=datacols,
    initial_species=inspecies,        # empty list is fine for a phi sweep
    common_properties=commonprop,
    os_input_file=inputstr,
    phi_inf=min(phi), phi_sup=max(phi),
    p_inf=P, p_sup=P,
    t_inf=T_in, t_sup=T_in,           # LFS: inf == sup == inlet temperature
    fuels_object=[sp_OME2.id],        # list of Species IDs (NOT the old `fuels` string field)
)

### Inspect before sending

In [ ]:
e.serialize()   # shows the JSON payload so you can sanity-check everything

### Send the experiment

In [ ]:
my_sciexpem.insertElement(e, verbose=True)

### After upload

- The experiment lands with `status='unverified'`.
- SciExpeM auto-matches the LFS interpreter **EXP24** (`PremixedLaminarFlame1D`)
  from the data-column name set `{composition, equivalence ratio, laminar burning velocity}`.
- Verify it and start a simulation against a kinetic model (see `launch-simul.ipynb`)
  to get the model LFS curve and curve-matching score.